# 00 · Warehouse, ETL vs ELT, and OLAP

**DMDW topics:** ETL vs ELT, star / snowflake schema, fact constellation, surrogate keys, OLAP operations (roll-up, drill-down, slice, dice, pivot), ROLLUP / CUBE / GROUPING SETS.

In [1]:
import pandas as pd
from IPython.display import Image, Markdown, display

from cfiq.config import load_config
from cfiq.db import get_engine, read_sql

cfg = load_config()
engine = get_engine()
REPORTS = cfg["paths"]["reports"]
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## ETL audit trail (`etl_metadata`)
Every phase appends one row per endpoint / table, so a run can be traced end to end.

In [2]:
read_sql(engine, '''
SELECT phase, endpoint, row_count, api_calls, cache_hits, status, finished_at
FROM etl_metadata
WHERE id IN (SELECT max(id) FROM etl_metadata GROUP BY phase, endpoint)
ORDER BY CASE phase WHEN 'extract' THEN 1 WHEN 'stage' THEN 2 WHEN 'transform' THEN 3 WHEN 'load' THEN 4 ELSE 5 END, id''')

,phase,endpoint,row_count,api_calls,cache_hits,status,finished_at
0,extract,contest.list,2172,0,1,success,2026-11-25 16:05:33.476518+00:00
1,extract,problemset.problems,11541,0,1,success,2026-11-25 16:05:33.502142+00:00
2,extract,user.ratedList,42145,0,1,success,2026-11-25 16:05:33.646230+00:00
3,extract,sample.stratified,350,0,0,success,2026-11-25 16:05:33.831178+00:00
4,extract,user.info,350,0,2,success,2026-11-25 16:05:33.833618+00:00
5,extract,user.rating,17390,0,350,success,2026-11-25 16:05:33.927579+00:00
6,extract,user.status,546417,0,350,success,2026-11-25 16:05:38.651297+00:00
7,stage,stg_contest,2172,0,0,success,2026-11-25 16:05:39.068132+00:00
8,stage,stg_problem,11541,0,0,success,2026-11-25 16:05:39.663193+00:00
9,stage,stg_user,350,0,0,success,2026-11-25 16:05:39.676243+00:00


## ELT: raw JSON lives in the database as jsonb

In [3]:
read_sql(engine, "SELECT handle, payload->>'verdict' AS verdict, payload->'problem'->>'name' AS problem, payload->'author'->>'participantType' AS participant FROM stg_submission LIMIT 5")

,handle,verdict,problem,participant
0,Abhinav_Dass,SKIPPED,0s and 1s,CONTESTANT
1,Abhinav_Dass,SKIPPED,Modulo and Counting,CONTESTANT
2,Abhinav_Dass,SKIPPED,Swap and Negate,CONTESTANT
3,Abhinav_Dass,SKIPPED,Lock and Unlock,CONTESTANT
4,Abhinav_Dass,SKIPPED,Math and Physics,CONTESTANT


In [4]:
display(Markdown((REPORTS / 'etl_vs_elt.md').read_text()))
display(Markdown((REPORTS / 'preprocessing.md').read_text()))

# ETL vs ELT: cleaning the submissions

| path | where the T runs | rows out | seconds |
|---|---|---:|---:|
| ETL | pandas, before loading (`cfiq/transform/clean.py`) | 148,643 | 12.89 |
| ELT | SQL inside Postgres on staged jsonb (`sql/elt_transform.sql`) | 148,643 | 1.21 |

Same row count from both paths: **yes**.

ETL time includes reading the raw JSON files from disk; ELT time is the SQL only (the raw JSON was already loaded into `stg_submission` by `cfiq stage`).
ETL keeps the warehouse clean and the logic in testable Python; ELT keeps the raw data in the database so new transformations can be written later in SQL without re-extracting.


# Preprocessing report

## Cleaning steps (submissions)

| step | rows |
|---|---:|
| raw submissions | 546,417 |
| after dedupe by id | 545,434 |
| after dropping gym / no contest | 477,257 |
| after keeping official participations | 148,643 |
| after dropping missing/in-flight verdicts | 148,643 |
| after joining problems on (contestId, index) | 148,643 |

## Missing values: problem rating

- Problems (problemset + integrated from submissions): **12,999**
- Missing `rating`: **489** (3.8%)
- Clean submissions on an unrated problem: **17,624**
- Strategy (a) impute: median rating of same division + index letter (fallback: index letter, then global), rounded to 100, flagged `rating_imputed`.
- Strategy (b) drop: remove those problems. The effect of each on the models is in `reports/classification/imputation_comparison.md`.

## Clean tables

| table | rows |
|---|---:|
| contests | 2,172 |
| problems | 12,999 |
| problem_tags | 36,759 |
| submissions | 148,643 |
| users | 350 |
| rating_changes | 17,390 |


## Star schema row counts

In [5]:
tables = ['dim_rating_band','dim_user','dim_contest','dim_problem','dim_tag_category','dim_tag','bridge_problem_tag','dim_language','dim_time','fact_submission','fact_rating_change']
pd.DataFrame({'table': tables, 'rows': [read_sql(engine, f'SELECT count(*) AS n FROM {t}')['n'][0] for t in tables]})

,table,rows
0,dim_rating_band,7
1,dim_user,350
2,dim_contest,2172
3,dim_problem,12999
4,dim_tag_category,7
5,dim_tag,39
6,bridge_problem_tag,36759
7,dim_language,63
8,dim_time,1567
9,fact_submission,148643


## OLAP queries (`sql/olap_queries.sql`)
Each query is run exactly as written in the SQL file.

In [6]:
from cfiq.warehouse.olap import OLAP_FILE, parse_named_queries
queries = parse_named_queries(OLAP_FILE.read_text())
list(queries)

['rollup_tag_to_category',
 'drilldown_category_to_tag',
 'slice_python',
 'dice_experts_cpp_python_recent',
 'pivot_category_by_band',
 'rollup_operator',
 'cube_operator',
 'grouping_sets_operator']

### Roll-up: tag → tag category

In [7]:
read_sql(engine, queries['rollup_tag_to_category'])

,category_name,submissions,accepted,acceptance_pct
0,math,66730,34545,51.77
1,strings,12014,6186,51.49
2,algorithms,109924,55347,50.35
3,implementation,63996,28661,44.79
4,data structures,23443,10434,44.51
5,graph theory,24767,9023,36.43
6,other,15767,2048,12.99


### Drill-down: category 'graph theory' → its tags, by year/month

In [8]:
read_sql(engine, queries['drilldown_category_to_tag']).head(15)

,category_name,tag_name,year,month,submissions,acceptance_pct
0,graph theory,2-sat,2014,9,17,17.65
1,graph theory,2-sat,2015,10,6,0.00
2,graph theory,2-sat,2017,2,6,100.00
3,graph theory,2-sat,2017,3,3,0.00
4,graph theory,2-sat,2017,10,3,100.00
5,graph theory,2-sat,2018,7,1,100.00
6,graph theory,2-sat,2018,12,6,0.00
7,graph theory,2-sat,2019,9,1,100.00
8,graph theory,2-sat,2019,10,7,28.57
9,graph theory,2-sat,2020,8,19,31.58


### Slice: language family = Python

In [9]:
read_sql(engine, queries['slice_python']).head(15)

,category_name,band_name,submissions,acceptance_pct
0,algorithms,Newbie,129,48.06
1,algorithms,Pupil,202,40.10
2,algorithms,Specialist,317,47.00
3,algorithms,Expert,1332,46.55
4,algorithms,Candidate Master,493,43.41
5,algorithms,Master,721,51.46
6,algorithms,Grandmaster,1975,41.87
7,data structures,Newbie,8,37.50
8,data structures,Pupil,18,33.33
9,data structures,Specialist,37,35.14


### Dice: bands {Expert, CM} × languages {C++, Python} × year ≥ 2024

In [10]:
read_sql(engine, queries['dice_experts_cpp_python_recent'])

,band_name,language_family,year,submissions,acceptance_pct
0,Expert,C++,2024,2811,46.64
1,Expert,C++,2025,5550,48.88
2,Expert,C++,2026,6585,52.77
3,Expert,Python,2024,427,58.78
4,Expert,Python,2025,107,27.10
5,Expert,Python,2026,368,47.28
6,Candidate Master,C++,2024,1734,52.88
7,Candidate Master,C++,2025,3239,48.75
8,Candidate Master,C++,2026,3358,57.50
9,Candidate Master,Python,2024,64,43.75


### Pivot: category × band cross-tab

In [11]:
read_sql(engine, queries['pivot_category_by_band'])

,category_name,newbie,pupil,specialist,expert,cand_master,master,grandmaster
0,algorithms,40.1,47.5,47.5,47.1,50.4,48.3,53.9
1,data structures,37.0,43.1,42.7,37.9,38.4,40.0,48.6
2,graph theory,23.9,31.0,32.1,32.9,30.1,33.2,39.2
3,implementation,47.8,49.3,47.5,43.6,44.8,43.8,44.5
4,math,46.8,50.0,49.6,49.5,52.2,48.7,54.8
5,other,8.9,10.6,12.6,12.4,13.5,15.5,12.7
6,strings,44.1,58.3,52.0,50.4,56.1,45.3,53.0


### ROLLUP(category, tag): subtotals along one hierarchy

In [12]:
read_sql(engine, queries['rollup_operator']).head(20)

,category,tag,submissions,acceptance_pct
0,algorithms,binary search,18517,41.80
1,algorithms,bitmasks,11682,44.28
2,algorithms,constructive algorithms,39822,48.27
3,algorithms,divide and conquer,4050,35.63
4,algorithms,dp,27443,46.69
5,algorithms,greedy,67028,52.24
6,algorithms,meet-in-the-middle,239,35.15
7,algorithms,sortings,18146,53.17
8,algorithms,ternary search,1030,37.48
9,algorithms,two pointers,10266,49.15


### CUBE(band, language family): all 4 grouping combinations

In [13]:
read_sql(engine, queries['cube_operator'])

,band,language_family,submissions,acceptance_pct
0,Newbie,C++,1937,41.66
1,Newbie,Java,192,52.08
2,Newbie,Python,181,50.83
3,Newbie,** ALL **,2321,43.08
4,Pupil,C++,9024,49.87
5,Pupil,Python,287,41.81
6,Pupil,Java,103,73.79
7,Pupil,C,43,23.26
8,Pupil,** ALL **,9471,49.71
9,Specialist,C++,10283,49.48


### GROUPING SETS: per band, per language, per month, and total

In [14]:
gs = read_sql(engine, queries['grouping_sets_operator'])
gs.groupby('grouped_by').head(5)

,grouped_by,member,submissions,acceptance_pct
0,band,Candidate Master,11117,50.06
1,band,Expert,22904,47.69
2,band,Grandmaster,67862,45.59
3,band,Master,24229,46.27
4,band,Newbie,2321,43.08
7,language,C,494,27.13
8,language,C#,683,52.12
9,language,C++,131481,46.95
10,language,Go,48,60.42
11,language,Java,1252,50.40
